Rework of Hybrid Control loop.
Here is for experiments and value validation

In [1]:
import time
import mujoco
import pinocchio as pino
import numpy as np
from hybrid_impedance import check_world_ee_contact_force

In [2]:
from utils import quick_plot
from hybrid_impedance import hierarchical_impedance_jacob

In [3]:
total_steps = 3

In [4]:
# Cartesian impedance control gains.
impedance_pos = np.asarray([100.0, 100.0, 100.0])  # [N/m]
impedance_ori = np.asarray([50.0, 50.0, 50.0])  # [Nm/rad]

# Joint impedance control gains.
Kp_null = np.asarray([75.0, 75.0, 50.0, 50.0, 40.0, 25.0, 25.0])

# Damping ratio for both Cartesian and joint impedance control.
damping_ratio = 1.0

# Whether to enable gravity compensation.
gravity_compensation: bool = True

# Simulation timestep in seconds.
dt: float = 0.002

In [26]:
# Load the model and data.
xml_path = "kuka_iiwa_14/scene_notarget.xml"
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)
pino_model = pino.buildModelFromMJCF(r"C:\wkspace\mj_ctrl\kuka_iiwa_14\iiwa14.xml")
pino_data = pino_model.createData()

model.opt.timestep = dt

key_name = "home"
key_id = model.key(key_name).id

In [27]:
pino_model.nv

7

In [6]:
# Compute damping and stiffness matrices.
damping_pos = damping_ratio * 2 * np.sqrt(impedance_pos)
damping_ori = damping_ratio * 2 * np.sqrt(impedance_ori)
Kp = np.concatenate([impedance_pos, impedance_ori], axis=0)
Kd = np.concatenate([damping_pos, damping_ori], axis=0)
Kd_null = damping_ratio * 2 * np.sqrt(Kp_null)

# End-effector site we wish to control.
site_name = "attachment_site"
site_id = model.site(site_name).id

# Get the dof and actuator ids for the joints we wish to control. These are copied
# from the XML file. Feel free to comment out some joints to see the effect on
# the controller.
joint_names = [
    "joint1",
    "joint2",
    "joint3",
    "joint4",
    "joint5",
    "joint6",
    "joint7",
]
dof_ids = np.array([model.joint(name).id for name in joint_names])
actuator_ids = np.array([model.actuator(name).id for name in joint_names])

# Initial joint configuration saved as a keyframe in the XML file.
key_name = "home"
key_id = model.key(key_name).id
q0 = model.key(key_name).qpos


target_pos = np.array([0.5, 0., 0.45])  # Note that the height of the table is 0.45m
target_quat = np.array([0., 1., 0., 0.])
x_dot_desired = np.zeros(2)
x_ddot_desired = np.zeros(2)

# Circle drawing parameters
circle_center = np.array([0.5, 0.0, 0.45])  # Center of circle on table
circle_radius = 0.1  # 10cm radius
circle_drawing = False
circle_duration = 10.0  # 10 seconds to complete one circle
contact_threshold = 8.0  # Force threshold to start drawing (close to desired 10N)
contact_stable_time = 0
contact_stable_duration = 1.0
angular_speed = np.pi / 4

# Pre-allocate numpy arrays.
jac = np.zeros((6, model.nv))
M_inv = np.zeros((model.nv, model.nv))
Mx = np.zeros((6, 6))

In [7]:
# Settings for the contact solver.
model.opt.cone = 0
# Visualize contact forces.
contact_forces = []

# Start when contacted

In [29]:
mujoco.mj_resetDataKeyframe(model, data, key_id)
saved_qpos = np.array([
    -3.12107748e-03,  3.52547424e-01,  1.39473444e-03,
    -1.67145666e+00,  1.47348211e-02,  1.17462243e+00,
    -7.49712653e-03
])
saved_qvel = np.array([
    -0.00150062, -0.00262153,  0.00036198,
    -0.00325501,  0.009429,   -0.02064151,
    -0.00510712
])
data.qpos[:] = saved_qpos
data.qvel[:] = saved_qvel
mujoco.mj_forward(model, data)


In [15]:
def dynamically_consistent_inv(jac, M_inv):
    """
    Compute dynamically consistent pseudoinverse
    J^{M+} = M^{-1} J^T (J M^{-1} J^T)^{-1}
    """
    Mx_inv = jac @ M_inv @ jac.T
    if abs(np.linalg.det(Mx_inv)) >= 1e-2:
        Mx = np.linalg.inv(Mx_inv)
    else:
        Mx = np.linalg.pinv(Mx_inv, rcond=1e-2)
    return M_inv @ jac.T @ Mx

In [44]:
# Main control loop
for step in range(total_steps):  # total_steps = int(total_duration / dt)
    step_start = time.time()
    current_contact_force = check_world_ee_contact_force(data, model)
    F_ext_phi = current_contact_force[2]
    F_ext_x = current_contact_force[:2]
    # desired trajectory
    elapsed_time = data.time - 0
    if elapsed_time < circle_duration:
        angle = angular_speed * elapsed_time
        # x
        target_pos[0] = circle_center[0] + circle_radius * np.cos(angle)
        target_pos[1] = circle_center[1] + circle_radius * np.sin(angle)
        target_pos[2] = circle_center[2]  # Keep Z at table height
        # x_dot
        x_dot_desired[0] = -circle_radius * angular_speed * np.sin(angle)
        x_dot_desired[1] =  circle_radius * angular_speed * np.cos(angle)
        # x_dot_desired[2] = 0.0
        # x_ddot
        x_ddot_desired[0] = -circle_radius * angular_speed**2 * np.cos(angle)
        x_ddot_desired[1] = -circle_radius * angular_speed**2 * np.sin(angle)
        # x_ddot_desired[2] = 0.0
    else:
        # Circle completed, stop drawing
        circle_drawing = False
        print("Circle drawing completed!")

    # Jacobian.
    mujoco.mj_jacSite(model, data, jac[:3], jac[3:], site_id)
    A = np.array([[0, 0, 1, 0, 0, 0]]) # 1 x 6
    B = np.array([[1, 0, 0, 0, 0, 0],   # 2 x 6
                [0, 1, 0, 0, 0, 0]])
    # jac 
    J_phi = jac[2:3, :]
    # J_motion = (2 * B.T @ B @ np.concatenate([data.site_xpos[site_id], np.zeros(3)])).T @ jac
    J_motion = jac[0:2, :]
    jac_1 = np.vstack([J_phi, J_motion]) # stacked phi and motion jacobi as one
    # according to paper equation (9), only null space Jacobian needs to be derived
    
    # Compute the task-space inertia matrix.
    mujoco.mj_solveM(model, data, M_inv, np.eye(model.nv))
    
    # dynamically consistent pseudoinverse
    jac_1_inv = dynamically_consistent_inv(jac_1, M_inv)
    N2 = np.eye(model.nv) - jac_1.T @ jac_1_inv.T
    # find null space J_null
    U, s, Vt = np.linalg.svd(jac_1)
    rank = np.sum(s > 1e-10)
    J_null = Vt[rank:, :] @ N2.T

    # Compute the task-space inertia matrix for x-y plane
    Mxy_inv = J_motion @ M_inv @ J_motion.T  # Now this will be 2x2
    if abs(np.linalg.det(Mxy_inv)) >= 1e-2:
        Mxy = np.linalg.inv(Mxy_inv)
    else:
        Mxy = np.linalg.pinv(Mxy_inv, rcond=1e-2)

    # task space
    x_tilde = data.site_xpos[site_id][0:2] - target_pos[0:2]
    site_vel = jac @ data.qvel[dof_ids] #[vx, vy, vz, wx, wy, wz]
    x_dot_tilde = site_vel[0:2] - x_dot_desired
    # F_ctrl_x = (Mxy @ x_ddot_desired + 
    #             C_x @ x_dot_desired - 
    #             K_x @ x_tilde - 
    #             D_x @ x_dot_tilde)
    # TODO：task space coriolis matrix in motion space is complicated 
    # https://www.perplexity.ai/search/mujoco-mj-solvem-model-data-m-qM7wE16wQSeuwmxFysbosw?5=d
    F_ctrl_x = (Mxy @ x_ddot_desired - 
                Kp[:2] * x_tilde - 
                Kd[:2] * x_dot_tilde)
    
    # TODO: null space, check this one, hierachial probably not to be used
    # Jbar = M_inv @ jac.T @ Mx
    F_ctrl_v = Kp_null * (q0 - data.qpos[dof_ids]) - Kd_null * data.qvel[dof_ids]
    # tau += (np.eye(model.nv) - jac.T @ Jbar.T) @ ddq
    
    # constraint space
    Mx_phi_inv = np.linalg.inv(J_phi @ M_inv @ J_phi.T)
    if abs(np.linalg.det(Mx_phi_inv)) >= 1e-2:
        lambda_phi = np.linalg.inv(Mx_phi_inv)
    else:
        lambda_phi = np.linalg.pinv(Mx_phi_inv, rcond=1e-2)
    C = pino.computeCoriolisMatrix(pino_model, pino_data, data.qpos, data.qvel) 
    F_desired_contact = np.array([-10.0])
    # computeJointJacobiansTimeVariation
    pino.computeJointJacobiansTimeVariation(pino_model, pino_data, data.qpos, data.qvel)
    J_dot = np.zeros((6, model.nv))
    J_dot = pino.getFrameJacobianTimeVariation(pino_model, pino_data, site_id, pino.LOCAL_WORLD_ALIGNED)
    J_phi_dot = A @ J_dot
    # J_null.T @ F_ctrl_v: how to get q_v??
    F_ctrl_constraint = (
        lambda_phi @ F_desired_contact -
        lambda_phi @ J_phi @ M_inv @ (J_motion.T @ F_ctrl_x + J_null.T @ F_ctrl_v) +
        lambda_phi @ J_phi @ M_inv @ (J_motion.T @ F_ext_x) +
        lambda_phi @ (J_phi @ M_inv @ C - J_phi_dot) @ data.qvel.copy()
    )
    # Set the control signal
    np.clip(tau, *model.actuator_ctrlrange.T, out=tau)
    data.ctrl[actuator_ids] = tau[actuator_ids]

    # Step the simulation
    mujoco.mj_step(model, data)

    # Real-time synchronization (optional)
    time_until_next_step = dt - (time.time() - step_start)
    if time_until_next_step > 0:
        time.sleep(time_until_next_step)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 7 is different from 4)

In [36]:
current_contact_force

np.float64(27.54372706705414)

In [33]:
J_dot = pino.getFrameJacobianTimeVariation(pino_model, pino_data, site_id, pino.LOCAL_WORLD_ALIGNED)

In [34]:
J_dot

array([[0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
elapsed_time = data.time - 0
if elapsed_time < circle_duration:
    # Calculate circle position
    angle = angular_speed * elapsed_time
    # x
    target_pos[0] = circle_center[0] + circle_radius * np.cos(angle)
    target_pos[1] = circle_center[1] + circle_radius * np.sin(angle)
    target_pos[2] = circle_center[2]  # Keep Z at table height
else:
    # Circle completed, stop drawing
    circle_drawing = False
    print("Circle drawing completed!")